# Week 4 – Milestone 1: Smart Healthcare Monitoring — Blockchain Ledger

This notebook connects Python to a local **Ganache** blockchain, loads the `HealthcareDataLedger` smart contract deployed via **Remix IDE**, stores patient health records on-chain, and verifies retrieval.

**Prerequisites:**
- Ganache desktop app is running (`http://127.0.0.1:7545`)
- `HealthcareDataLedger.sol` deployed in Remix IDE
- `web3` and `pandas` installed

---
## Step 1 – Install Dependencies

In [7]:
# Run this cell once if packages are not yet installed
# !pip install web3 pandas numpy

---
## Step 2 – Connect to Ganache

In [21]:
from web3 import Web3

# Connect to local Ganache blockchain
ganache_url = "http://127.0.0.1:7545"
web3 = Web3(Web3.HTTPProvider(ganache_url))

if web3.is_connected():
    print("Connected to Ganache successfully!")
    print(f"   Chain ID : {web3.eth.chain_id}")
    print(f"   Block #  : {web3.eth.block_number}")
    print(f"   Accounts : {len(web3.eth.accounts)} available")
else:
    print("Connection failed. Ensure Ganache is running.")

Connected to Ganache successfully!
   Chain ID : 1337
   Block #  : 14
   Accounts : 10 available


---
## Step 3 – Load the Smart Contract

Paste your **contract address** and **ABI** from Remix IDE below.

In [22]:
contract_address = "0xb5D4e55e99b5BFa079ba214722E44d4CbD5e7C81"

abi = [
  {
    "inputs": [
      {
        "internalType": "string",
        "name": "patientId",
        "type": "string"
      },
      {
        "internalType": "uint256",
        "name": "heartRate",
        "type": "uint256"
      },
      {
        "internalType": "string",
        "name": "bloodPressure",
        "type": "string"
      },
      {
        "internalType": "uint256",
        "name": "oxygenLevel",
        "type": "uint256"
      },
      {
        "internalType": "string",
        "name": "bodyTemp",
        "type": "string"
      }
    ],
    "name": "storeData",
    "outputs": [],
    "stateMutability": "nonpayable",
    "type": "function"
  },
  {
    "inputs": [
      {
        "internalType": "uint256",
        "name": "index",
        "type": "uint256"
      }
    ],
    "name": "getRecord",
    "outputs": [
      {
        "internalType": "string",
        "name": "patientId",
        "type": "string"
      },
      {
        "internalType": "uint256",
        "name": "heartRate",
        "type": "uint256"
      },
      {
        "internalType": "string",
        "name": "bloodPressure",
        "type": "string"
      },
      {
        "internalType": "uint256",
        "name": "oxygenLevel",
        "type": "uint256"
      },
      {
        "internalType": "string",
        "name": "bodyTemp",
        "type": "string"
      },
      {
        "internalType": "uint256",
        "name": "timestamp",
        "type": "uint256"
      }
    ],
    "stateMutability": "view",
    "type": "function"
  },
  {
    "inputs": [],
    "name": "getTotalRecords",
    "outputs": [
      {
        "internalType": "uint256",
        "name": "",
        "type": "uint256"
      }
    ],
    "stateMutability": "view",
    "type": "function"
  }
]


# ───────────────────────────────────────────────────────────────────────────────

contract_address = Web3.to_checksum_address(contract_address)
contract = web3.eth.contract(address=contract_address, abi=abi)
web3.eth.default_account = web3.eth.accounts[0]

print(f"Connected to Smart Contract at {contract_address}")
print(f"   Default sender: {web3.eth.default_account}")

Connected to Smart Contract at 0xb5D4e55e99b5BFa079ba214722E44d4CbD5e7C81
   Default sender: 0xA5b27fd95F76CFD443c5B0E32d021Cac5E7fF1DF


---
## Step 4 – Verify Contract is Responding

In [23]:
total_records = contract.functions.getTotalRecords().call()
print(f" Contract responding. Total records on-chain: {total_records}")

 Contract responding. Total records on-chain: 8


---
## Step 5 – Store a Dummy Patient Record (Transaction Test)

In [25]:
print("Sending storeData() transaction...")

txn = contract.functions.storeData(
    "TEST001",   # patientId
    72,          # heartRate
    "120/80",   # bloodPressure
    98,          # oxygenLevel
    "36.6"       # bodyTemp
).transact({
    'from': web3.eth.default_account,
    'gas': 1000000
})

receipt = web3.eth.wait_for_transaction_receipt(txn)
print("Dummy data stored on blockchain!")
print(f"   Tx Hash    : {txn.hex()}")
print(f"   Block #    : {receipt['blockNumber']}")
print(f"   Gas Used   : {receipt['gasUsed']}")

Sending storeData() transaction...
Dummy data stored on blockchain!
   Tx Hash    : 9660ce167b97cb5ae82be62694ae18690521bab831e0fea45b93097265533acc
   Block #    : 16
   Gas Used   : 165333


---
## Step 6 – Verify Data Retrieval

In [16]:
total_records = contract.functions.getTotalRecords().call()
print(f"Total Records: {total_records}")

record = contract.functions.getRecord(0).call()
print("\nFirst Stored Record:")
print(f"  Patient ID     : {record[0]}")
print(f"  Heart Rate     : {record[1]} bpm")
print(f"  Blood Pressure : {record[2]}")
print(f"  Oxygen Level   : {record[3]}%")
print(f"  Body Temp      : {record[4]}°C")

import datetime
ts = datetime.datetime.fromtimestamp(record[5], datetime.timezone.utc)
print(f"  Timestamp      : {ts} UTC")

Total Records: 7

First Stored Record:
  Patient ID     : TEST001
  Heart Rate     : 72 bpm
  Blood Pressure : 120/80
  Oxygen Level   : 98%
  Body Temp      : 36.6°C
  Timestamp      : 2026-05-21 10:47:11+00:00 UTC


---
## Step 7 – Store Sample Healthcare Records from Week 2 Dataset

In [13]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Recreate the Week 2 dataset
num_records = 5  # Store 5 records as a demo (increase if needed)
data = []
for _ in range(num_records):
    record = {
        "patient_id":     f"PAT{np.random.randint(100, 999)}",
        "heart_rate":     int(np.random.randint(60, 100)),
        "blood_pressure": f"{np.random.randint(100,140)}/{np.random.randint(60,90)}",
        "oxygen_level":   int(np.random.randint(95, 100)),
        "body_temp":      str(round(np.random.uniform(36.0, 38.0), 1))
    }
    data.append(record)

df = pd.DataFrame(data)
print("Sample patient records to store:")
print(df.to_string(index=False))

Sample patient records to store:
patient_id  heart_rate blood_pressure  oxygen_level body_temp
    PAT994          78         138/74            96      36.2
    PAT753          93         130/70            96      36.0
    PAT887          77         110/84            95      36.8
    PAT553          96         122/79            98      36.1
    PAT976          93         117/74            99      36.1


In [26]:
# Store each patient record on the blockchain
for _, row in df.iterrows():
    txn = contract.functions.storeData(
        row['patient_id'],
        row['heart_rate'],
        row['blood_pressure'],
        row['oxygen_level'],
        row['body_temp']
    ).transact({'from': web3.eth.default_account, 'gas': 1000000})
    web3.eth.wait_for_transaction_receipt(txn)
    print(f"Stored {row['patient_id']} | HR:{row['heart_rate']} | BP:{row['blood_pressure']} | O2:{row['oxygen_level']}% | Temp:{row['body_temp']}°C")

total = contract.functions.getTotalRecords().call()
print(f"\nTotal records now on-chain: {total}")

Stored PAT994 | HR:78 | BP:138/74 | O2:96% | Temp:36.2°C
Stored PAT753 | HR:93 | BP:130/70 | O2:96% | Temp:36.0°C
Stored PAT887 | HR:77 | BP:110/84 | O2:95% | Temp:36.8°C
Stored PAT553 | HR:96 | BP:122/79 | O2:98% | Temp:36.1°C
Stored PAT976 | HR:93 | BP:117/74 | O2:99% | Temp:36.1°C

Total records now on-chain: 15


---
## Step 8 – Display All Records from Blockchain

In [17]:
import datetime

total = contract.functions.getTotalRecords().call()
all_records = []

for i in range(total):
    r = contract.functions.getRecord(i).call()
    all_records.append({
        "Index":          i,
        "Patient ID":     r[0],
        "Heart Rate":     f"{r[1]} bpm",
        "Blood Pressure": r[2],
        "Oxygen Level":   f"{r[3]}%",
        "Body Temp":      f"{r[4]}°C",
        "Timestamp (UTC)": datetime.datetime.fromtimestamp(r[5], datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    })

result_df = pd.DataFrame(all_records)
print(f"All {total} records retrieved from blockchain:")
result_df

All 7 records retrieved from blockchain:


,Index,Patient ID,Heart Rate,Blood Pressure,Oxygen Level,Body Temp,Timestamp (UTC)
0,0,TEST001,72 bpm,120/80,98%,36.6°C,2026-05-21 10:47:11
1,1,TEST001,72 bpm,120/80,98%,36.6°C,2026-05-21 10:47:33
2,2,PAT994,78 bpm,138/74,96%,36.2°C,2026-05-21 10:50:36
3,3,PAT753,93 bpm,130/70,96%,36.0°C,2026-05-21 10:50:36
4,4,PAT887,77 bpm,110/84,95%,36.8°C,2026-05-21 10:50:36
5,5,PAT553,96 bpm,122/79,98%,36.1°C,2026-05-21 10:50:36
6,6,PAT976,93 bpm,117/74,99%,36.1°C,2026-05-21 10:50:36


---
## Summary

| Step | Action | Status |
|------|--------|--------|
| 1 | Installed `web3`, `pandas`, `numpy` | ✅ |
| 2 | Connected to Ganache at `http://127.0.0.1:7545` | ✅ |
| 3 | Loaded `HealthcareDataLedger` contract from Remix | ✅ |
| 4 | Verified contract responds to `getTotalRecords()` | ✅ |
| 5 | Stored dummy patient record via `storeData()` | ✅ |
| 6 | Retrieved and printed first stored record | ✅ |
| 7 | Stored 5 sample records from Week 2 dataset | ✅ |
| 8 | Displayed all blockchain records as a table | ✅ |